In [18]:
import math
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter,defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import Dataset,Subset,DataLoader,random_split
from tqdm.notebook import tqdm
from torch.optim.lr_scheduler import ReduceLROnPlateau
import itertools
import regex as re
import pickle

In [19]:
from google.colab import drive
import os,sys
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
%%script echo skip

text="aaabaa-kra rkxassaab"
print(list(text.encode("utf-8")))

skip


In [21]:
def count_pairs(ids, counts=None):
  if counts is None:
    counts=defaultdict(int)
  for pair in zip(ids,ids[1:]):
    counts[pair]+=1

  return counts

In [22]:
%%script echo skip

print(count_pairs(list(text.encode("utf-8"))))

skip


In [23]:
def merge(ids,pair,new_id):
  merge_ids=[]
  i=0

  while i<len(ids):
    if i<len(ids)-1 and (ids[i], ids[i+1]) == pair:
      merge_ids.append(new_id)
      i+=2
    else:
      merge_ids.append(ids[i])
      i+=1

  return merge_ids

In [24]:
def pretokenize(text):
  pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
  return re.findall(pattern,text)

'(?:[sdmt]|ll|ve|re) : 영어 축약형 ('s, 'd, 'm, 't,  'll, 've, 're)

?\p{L}+ : 단어 (앞에 공백 가능)

?\p{N}+ : 연속된 숫자

?[^\s\p{L}\p{N}]+ : 구두점 및 기호

\s+(?!\S)|\s+ : 공백 문자


findall(pattern,text) (text에서 pattern에 해당하는 부분 다 찾기)

In [25]:
def train_bpe(input_text, vocab_size, end_token="<|endoftext|>"):

  ids_list=[]
  for text in input_text.split(end_token):
    for pretoken in pretokenize(text):
      ids_list.append(list(pretoken.encode("utf-8")))

  num_merges=vocab_size-256-1
  merge_rules={}

  for step in tqdm(range(num_merges),desc="Training BPE"):
    counts=defaultdict(int)

    for ids in ids_list:
      counts= count_pairs(ids,counts)

    if not counts:
      break

    best_pair=max(counts,key=counts.get)
    new_id=step+256

    merge_rules[best_pair]=new_id

    for i in range(len(ids_list)):
      ids_list[i]=merge(ids_list[i],best_pair,new_id)

  return merge_rules

In [26]:
%%script echo skip

txt="hello world!<|endoftoken|>Nice to meet you"

merge_rules=train_bpe(txt,vocab_size=260)
print(merge_rules)

skip


In [27]:
%%script echo skip

print(bytes([11])+bytes([16]))

skip


In [28]:
class BPETokenizer:
  def __init__(self, merge_rules,end_token="<|endoftext|>"):
    self.merge_rules=merge_rules
    self.end_token=end_token
    self.end_token_id=256+len(merge_rules)

    self.id_to_bytes = {i: bytes([i]) for i in range(256)}

    for (id1,id2), new_id in merge_rules.items():
      self.id_to_bytes[new_id]=self.id_to_bytes[id1]+self.id_to_bytes[id2]

    self.id_to_bytes[self.end_token_id]=self.end_token.encode("utf-8")
    self.vocab_size=len(self.id_to_bytes)

  @staticmethod
  def load_from(filepath):
    with open(filepath,"rb") as f:
      merge_rules=pickle.load(f)

    return BPETokenizer(merge_rules)

  def _encode_text(self,text):
    ids=list(text.encode("utf-8"))

    #먼저 넣은 순서대로 돌아감
    for merge_pair, new_id in self.merge_rules.items():
      ids=merge(ids,merge_pair,new_id)

    return ids

  def encode(self,input_text,show_progress=False):
    pattern="("+re.escape(self.end_token)+")"
    texts=re.split(pattern,input_text)
    all_ids=[]

    texts=tqdm(texts, desc="Encoding") if show_progress else texts

    for text in texts:
      if text==self.end_token:
        all_ids.append(self.end_token_id)
      else:
        for pretoken in pretokenize(text):
          all_ids.extend(self._encode_text(pretoken))

    return all_ids

  def decode(self,ids):
    byte_list=[self.id_to_bytes[i] for i in ids]
    text_bytes=b"".join(byte_list)

    return text_bytes.decode("utf-8",errors="replace")

re.split(pattern,input) 에서 pattern에다가 괄호 씌워야 end_token도 같이 text로 넣을 수 있음. 안 그러면 end_token 빼고 split 됨

extend는 append에서 list 껍데기를 빼고 이어붙이는거라 보면 됨

In [29]:
%%script echo skip

tokenizer=BPETokenizer(merge_rules)
ids=tokenizer.encode(txt)

print(ids)
print(tokenizer.decode(ids))

skip


In [30]:
%%script echo skip

sample_text="Oh, and in case I don't see you.<|endoftext|>Good afternoon, Good evening and Good night."

merge_rules= train_bpe(sample_text, vocab_size=280)
tokenizer=BPETokenizer(merge_rules)

text="Good afternoon, Good evening and Good night."
ids=tokenizer.encode(text)
decoded=tokenizer.decode(ids)

print(ids)
print(decoded)

for token_id in ids:
  decoded_token=tokenizer.decode([token_id])
  print(f"{token_id} -> '{decoded_token}'")

skip


In [40]:
os.chdir('/content/drive/MyDrive/DeepLerning (military)/Deep-Learning-Study/DLFS6/BPE')

txt_data_path='tiny_codes.txt'
vocab_size=1000

In [41]:
with open(txt_data_path, 'r') as f:
  text = f.read()

merge_rules = train_bpe(text, vocab_size)

Training BPE:   0%|          | 0/743 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
with open('merge_rules_tiny_codes.pkl', 'wb') as f:
  pickle.dump(merge_rules, f)

In [38]:
tokenizer=BPETokenizer.load_from('merge_rules_tiny_codes.pkl')

In [39]:
print("처음에 학습된 10개")
for token_id in range(256,266):
  byte_seq=tokenizer.id_to_bytes[token_id]
  decoded_token=byte_seq.decode("utf-8")
  print(f" ID {token_id}: '{decoded_token}'")

print("\n마지막에 학습된 10개")
for token_id in range(990,1000):
  byte_seq=tokenizer.id_to_bytes[token_id]
  decoded_token=byte_seq.decode("utf-8")
  print(f" ID {token_id}: '{decoded_token}'")

처음에 학습된 10개
 ID 256: '  '
 ID 257: '    '
 ID 258: 'in'
 ID 259: '   '
 ID 260: 're'
 ID 261: 'or'
 ID 262: ' ='
 ID 263: 'st'
 ID 264: 'te'
 ID 265: '
   '

마지막에 학습된 10개
 ID 990: 'Error'
 ID 991: ' '''
 ID 992: 'atter'
 ID 993: 'St'
 ID 994: 'thod'
 ID 995: ' on'
 ID 996: 'cle'
 ID 997: ' R'
 ID 998: 'ery'
 ID 999: '<|endoftext|>'


In [37]:
import shutil

dst = '/content/drive/MyDrive/DeepLerning (military)/Deep-Learning-Study/DLFS6/BPE/'
shutil.copy('/content/merge_rules_tiny_codes.pkl', dst)

'/content/drive/MyDrive/DeepLerning (military)/Deep-Learning-Study/DLFS6/BPE/merge_rules_tiny_codes.pkl'